# Pingüinos como termómetros: ¿qué nivel explica la variabilidad morfológica?
## Drill-down jerárquico + descomposición de varianza — especie, isla, sexo e individuo

**Pregunta central:** En el dataset Palmer Penguins, ¿qué proporción de la variabilidad de rasgos morfométricos (p. ej. masa corporal) se asocia a especie, isla, sexo y año, y cómo cambia la interpretación al desagregar jerárquicamente en un diseño no balanceado?

---

## 1. Contexto: observación de campo y estructura jerárquica (½ cuartilla)

2007–2009. Palmer Station LTER, Península Antártica. Investigadores en zodiacs se acercan a colonias de pingüinos, miden picos con calibre digital, pesan adultos, extraen una gota de sangre para sexar. No es un experimento de laboratorio: es **ciencia de campo en condiciones extremas**.

El dataset de Palmer Penguins (`penguins.csv`) registra observaciones individuales con variables morfométricas y categóricas clave: `species` (Adelie, Chinstrap, Gentoo), `island` (Biscoe, Dream, Torgersen), `sex`, `year`, longitud/profundidad de pico, longitud de aleta y masa corporal. Esta estructura permite contar una historia jerárquica: cada individuo pertenece a una especie y fue medido en una isla y año concretos, pero el muestreo no es balanceado ni ortogonal entre factores.

Los pingüinos son **indicadores centinela del cambio climático** (Trivelpiece et al. 2011). Cambios en masa corporal y morfología del pico reflejan disponibilidad de krill, temperatura oceánica y cobertura de hielo. Aquí el problema no es el clima directamente: es **desenredar jerarquías**. ¿La variación que vemos se debe principalmente a diferencias entre especies, al lugar (isla), al dimorfismo sexual o a variabilidad individual? El drill-down jerárquico y la descomposición de varianza son las herramientas adecuadas para responder sin confundir niveles.

**Hipótesis formales:**
- H₀: σ²_especie = σ²_isla (especie e isla explican igual varianza)
- H₁: σ²_especie > σ²_isla (especie explica más varianza morfológica)

**Parámetros de interés:** ICC (intraclass correlation) en cada nivel; η² parcial para sexo; variance partitioning σ²_isla, σ²_especie, σ²_sexo, σ²_año y σ²_residual.

## 2. Dataset: Palmer Station LTER 2007–2009 (½ cuartilla)

In [ ]:
# Carga
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv(Path("../data/raw/penguins.csv"))
print(f"Shape: {df.shape}")
print(f"Missing:\n{df.isnull().sum()}")
print(f"\nEspecies: {df.species.unique()}")
print(f"Islas: {df.island.unique()}")
print(f"Años: {sorted(df.year.unique())}")
print(f"Sexo: {df.sex.unique()}")

# Tabla de contingencia especie × isla
ct = pd.crosstab(df.species, df.island, margins=True)
print(f"\nEspecie × Isla:\n{ct}")

# Stats por grupo
group_stats = df.groupby(['island', 'species'])['body_mass_g'].agg(['count', 'mean', 'std']).round(1)
print(f"\nMasa corporal (g) por isla/especie:\n{group_stats}")

**Fuente:** Palmer Station LTER (Long Term Ecological Research), datos 2007–2009 (Gorman et al. 2014).
**Tamaño:** 344 observaciones, 3 especies (Adelie, Chinstrap, Gentoo), 3 islas (Torgersen, Biscoe, Dream), 3 años.
**Variables:** `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`, `year`.
**Missingness:** 11 filas con `sex` NaN (2.9%); 2 filas con NaN en medidas morfológicas.
**Diseño incompleto (crítico — esto cambia todo el modelo):**
- **Biscoe: solo Gentoo** (n=124)
- **Dream:** Adelie (n=56) + Chinstrap (n=68)
- **Torgersen: solo Adelie** (n=52)

**Sesgos:**
- Muestreo no aleatorio: solo adultos reproductores en colonias accesibles
- Solo 3 años: variabilidad interanual subestimada
- Sexo missing not at random (difícil sexar en campo sin manipulación invasiva)

## 3. El drill-down: Isla → Especie → Individuo (¾ cuartilla)

In [ ]:
import sys
from pathlib import Path
_ROOT = Path('..').resolve()
for _p in [str(_ROOT), str(_ROOT / 'src')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# 1. Violin plots: body_mass_g por especie (color) faceteado por isla
import matplotlib.pyplot as plt
import seaborn as sns
from src.viz import setup_plotting_style, save_fig
sys.path.append("../src")
setup_plotting_style()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Masa corporal
sns.violinplot(data=df, x='species', y='body_mass_g', hue='island', split=False,
               inner='quartile', ax=axes[0], palette='Set2')
axes[0].set_title('Masa corporal por especie e isla')
axes[0].set_ylabel('Masa corporal (g)')

# Longitud pico
sns.violinplot(data=df, x='species', y='bill_length_mm', hue='island', split=False,
               inner='quartile', ax=axes[1], palette='Set2')
axes[1].set_title('Longitud pico por especie e isla')
axes[1].set_ylabel('Longitud pico (mm)')

plt.tight_layout()
save_fig("04_violin_mass_bill")
plt.show()

In [ ]:
# 2. Pairwise comparisons: Tukey HSD especie, isla, interacción
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import f_oneway

# ANOVA simple: especie
species_groups = [g['body_mass_g'].dropna().values for n, g in df.groupby('species')]
f_sp, p_sp = f_oneway(*species_groups)
print(f"ANOVA especie: F={f_sp:.2f}, p={p_sp:.2e}")
tukey_sp = pairwise_tukeyhsd(df['body_mass_g'].dropna(), df.loc[df['body_mass_g'].notna(), 'species'])
print(tukey_sp.summary())

# ANOVA simple: isla
island_groups = [g['body_mass_g'].dropna().values for n, g in df.groupby('island')]
f_isl, p_isl = f_oneway(*island_groups)
print(f"\nANOVA isla: F={f_isl:.2f}, p={p_isl:.2e}")
tukey_isl = pairwise_tukeyhsd(df['body_mass_g'].dropna(), df.loc[df['body_mass_g'].notna(), 'island'])
print(tukey_isl.summary())

In [ ]:
# 3. ICC 'naked': var(group_means)/var(total)
from stats_helpers import icc_naked

icc_species = icc_naked(df.dropna(subset=['body_mass_g']), 'body_mass_g', 'species')
icc_island = icc_naked(df.dropna(subset=['body_mass_g']), 'body_mass_g', 'island')
icc_year = icc_naked(df.dropna(subset=['body_mass_g']), 'body_mass_g', 'year')

print(f"ICC especie (body_mass_g): {icc_species:.4f} ({icc_species*100:.1f}% varianza entre especies)")
print(f"ICC isla (body_mass_g): {icc_island:.4f} ({icc_island*100:.1f}% varianza entre islas)")
print(f"ICC año (body_mass_g): {icc_year:.4f} ({icc_year*100:.1f}% varianza entre años)")

# Para bill_length_mm
icc_species_bill = icc_naked(df.dropna(subset=['bill_length_mm']), 'bill_length_mm', 'species')
icc_island_bill = icc_naked(df.dropna(subset=['bill_length_mm']), 'bill_length_mm', 'island')
print(f"\nICC especie (bill_length_mm): {icc_species_bill:.4f}")
print(f"ICC isla (bill_length_mm): {icc_island_bill:.4f}")

In [ ]:
# 4. Visual drill-down: Treemap jerárquico (isla → especie → individuo)
import plotly.express as px

# Preparar datos para treemap: contar individuos por isla/especie
treemap_data = df.groupby(['island', 'species']).size().reset_index(name='count')
treemap_data['path'] = treemap_data['island'] + ' / ' + treemap_data['species']

fig = px.treemap(treemap_data, path=['island', 'species'], values='count',
                 color='species', color_discrete_sequence=px.colors.qualitative.Set2,
                 title='Drill-down: Isla → Especie → Individuo (n=344)')
fig.update_traces(textinfo='label+value+percent parent+percent root')
fig.write_html("../reports/figures/04_treemap.html")
fig.show()

In [ ]:
# 5. Efecto sexo (donde disponible)
df_sex = df.dropna(subset=['sex', 'body_mass_g'])

from stats_helpers import efecto_tamano_cohen_d, t_test_welch

male = df_sex[df_sex.sex == 'MALE']['body_mass_g'].values
female = df_sex[df_sex.sex == 'FEMALE']['body_mass_g'].values

d = efecto_tamano_cohen_d(male, female)
ttest = t_test_welch(male, female)

print(f"Masa corporal: Machos={male.mean():.0f}g, Hembras={female.mean():.0f}g")
print(f"Cohen's d: {d:.3f}")
print(f"t-test: diff={ttest['diff']:.1f}g, IC95%=[{ttest['ci'][0]:.1f}, {ttest['ci'][1]:.1f}], p={ttest['p']:.2e}")

> **Lectura guiada por la evidencia:** el drill-down (agregado → por isla → por especie → por sexo/individuo) y las métricas de partición de varianza (ICC/η² según especificación) permiten distinguir qué parte de las diferencias observadas se asocia a taxonomía, a contexto espacial/temporal o a variación intra-grupo. Dado el diseño incompleto (no todas las especies en todas las islas), los contrastes deben interpretarse con cautela respecto a confusión entre factores.

**Hallazgos a extraer de las salidas:**
1. **Eje dominante:** contribución relativa de especie frente a isla/año en los rasgos analizados.
2. **Dimorfismo sexual:** magnitud y dirección de diferencias por sexo (tamaños de efecto) cuando hay datos.
3. **Diseño y confusión:** qué combinaciones isla–especie están presentes y cómo limita la separación de efectos.
4. **Robustez:** consistencia entre descriptivos, modelos/ANOVA y validación por grupos (p. ej. leave-one-island-out) cuando se calcula.

In [ ]:
# Registrar hallazgos (dinámicos, basados en cálculos del notebook)
from narrative import hallazgo
from IPython.display import display, Markdown
import numpy as np
import pandas as pd

# Recalcular ICCs "naked" para masa corporal (independiente de sobrescrituras posteriores)
from stats_helpers import icc_naked, efecto_tamano_cohen_d, t_test_welch

df_mass = df.dropna(subset=['body_mass_g'])
icc_sp_n = icc_naked(df_mass, 'body_mass_g', 'species')
icc_is_n = icc_naked(df_mass, 'body_mass_g', 'island')
icc_yr_n = icc_naked(df_mass, 'body_mass_g', 'year')

display(hallazgo(
    f"Partición descriptiva de varianza (ICC 'naked', masa corporal): especie = {icc_sp_n:.3f} ({icc_sp_n*100:.1f}%), isla = {icc_is_n:.3f} ({icc_is_n*100:.1f}%), año = {icc_yr_n:.3f} ({icc_yr_n*100:.1f}%).",
    evidencia="ICC no condicionado: var(medias de grupo)/var(total) sobre body_mass_g",
    ic=None
))

# Sexo
df_sex = df.dropna(subset=['sex', 'body_mass_g'])
if len(df_sex) and df_sex['sex'].nunique() >= 2:
    male = df_sex[df_sex.sex == 'MALE']['body_mass_g'].values
    female = df_sex[df_sex.sex == 'FEMALE']['body_mass_g'].values
    d_sex = efecto_tamano_cohen_d(male, female)
    t_sex = t_test_welch(male, female)
    display(hallazgo(
        f"Dimorfismo sexual en masa corporal (observaciones con sexo registrado): media machos = {np.mean(male):.1f} g, hembras = {np.mean(female):.1f} g; diferencia = {t_sex['diff']:.1f} g (IC95% [{t_sex['ci'][0]:.1f}, {t_sex['ci'][1]:.1f}], p={t_sex['p']:.3g}); Cohen's d = {d_sex:.3f}.",
        evidencia="Comparación por sexo (Welch) sobre registros completos de sex y body_mass_g",
        ic=(float(t_sex['ci'][0]), float(t_sex['ci'][1])),
        pval=float(t_sex['p']) if np.isfinite(t_sex['p']) else None
    ))

# Diseño incompleto / confusión isla-especie
ct = pd.crosstab(df['species'], df['island'])
present = int((ct.values > 0).sum())
total_cells = int(ct.shape[0] * ct.shape[1])
# Medida simple de asociación nominal (Cramér's V) sin dependencias extra
try:
    from scipy.stats import chi2_contingency
    chi2, p_chi, dof, _ = chi2_contingency(ct.values)
    n_obs = ct.values.sum()
    cramers_v = np.sqrt(chi2 / (n_obs * (min(ct.shape)-1))) if n_obs > 0 and min(ct.shape) > 1 else np.nan
    assoc_txt = f"Cramér's V = {cramers_v:.3f} (chi2={chi2:.2f}, p={p_chi:.3g})"
except Exception:
    assoc_txt = "asociación fuerte esperable por diseño no balanceado"
display(hallazgo(
    f"Diseño incompleto especie×isla: {present} de {total_cells} combinaciones presentes en los datos. Esto implica confusión parcial entre factores y limita separar efectos puros de isla y especie; {assoc_txt}.",
    evidencia="Tabla de contingencia species×island",
    ic=None
))


## 4. Modelo mixto: especies fijas, islas aleatorias (1 cuartilla)

In [ ]:
# Mixed model con crossed random effects (diseño incompleto → NO nested)
# body_mass_g ~ sex + (1|island) + (1|species) + (1|year)
from stats_helpers import mixed_model_variance_partition

df_clean = df.dropna(subset=['body_mass_g', 'sex', 'island', 'species', 'year']).copy()

# Modelo: fixed effect sexo, random intercepts isla, especie, año
model_mm = mixed_model_variance_partition(
    df_clean,
    'body_mass_g ~ sex',
    groups='island',  # agrupamiento principal para RE
    re_formula='~1'
)
print("=== MIXED MODEL: sex + (1|island) ===")
print(f"Fixed effects: {model_mm['fixed_effects']}")
print(f"Var components: {model_mm['var_components']}")
icc_island = float(np.asarray(model_mm['icc']).ravel()[0])
print(f"ICC island: {icc_island:.4f}")

# Modelo alternativo: species como random effect
model_mm2 = mixed_model_variance_partition(
    df_clean,
    'body_mass_g ~ sex',
    groups='species',
    re_formula='~1'
)
print(f"\n=== MIXED MODEL: sex + (1|species) ===")
print(f"Fixed effects: {model_mm2['fixed_effects']}")
print(f"Var components: {model_mm2['var_components']}")
icc_species = float(np.asarray(model_mm2['icc']).ravel()[0])
print(f"ICC species: {icc_species:.4f}")

In [ ]:
# Modelo con crossed random effects usando statsmodels MixedLM directo
# body_mass_g ~ sex + (1|island) + (1|species) + (1|year)
import statsmodels.formula.api as smf

# Statsmodels no soporta crossed RE nativamente fácilmente
# Usamos especie como fixed effect + isla + año como random (diseño incompleto)
model_fe = smf.mixedlm('body_mass_g ~ sex + C(species)', df_clean, groups=df_clean['island'], re_formula='1')
result_fe = model_fe.fit(reml=True)
print("=== SPECIES FIXED + ISLAND RANDOM ===")
print(result_fe.summary().tables[1])
print(f"\nVar island: {result_fe.cov_re.iloc[0,0]:.2f}")
print(f"Var residual: {result_fe.scale:.2f}")
icc_island_adj = result_fe.cov_re.iloc[0,0] / (result_fe.cov_re.iloc[0,0] + result_fe.scale)
print(f"ICC island (ajustado por especie): {icc_island_adj:.4f}")

In [ ]:
# Variance partitioning manual (ANOVA tipo III para mixed)
from statsmodels.stats.anova import anova_lm

# Modelo OLS saturado para effect sizes
ols_full = smf.ols('body_mass_g ~ C(island) + C(species) + C(sex) + C(year)', df_clean).fit()
anova_table = anova_lm(ols_full, typ=3)
print(anova_table)

# η² parcial
ss_total = anova_table['sum_sq'].sum()
anova_table['eta2_partial'] = anova_table['sum_sq'] / (anova_table['sum_sq'] + anova_table.loc['Residual', 'sum_sq'])
print(f"\nη² parcial:\n{anova_table[['eta2_partial']].round(4)}")

In [ ]:
# Validación: Leave-One-Island-Out CV
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

X = pd.get_dummies(df_clean[['sex', 'species', 'year']], drop_first=True)
y = df_clean['body_mass_g'].values
groups = df_clean['island'].values

logo = LeaveOneGroupOut()
preds, actuals = [], []

for train_idx, test_idx in logo.split(X, y, groups):
    model = LinearRegression()
    model.fit(X.iloc[train_idx], y[train_idx])
    pred = model.predict(X.iloc[test_idx])
    preds.extend(pred)
    actuals.extend(y[test_idx])

rmse = np.sqrt(mean_squared_error(actuals, preds))
r2 = r2_score(actuals, preds)
print(f"LOO-Island CV: RMSE={rmse:.1f}g, R²={r2:.3f}")

# Predicción por isla
for island in df_clean['island'].unique():
    mask = df_clean['island'] == island
    if mask.sum() > 0:
        r2_i = r2_score(y[mask], np.array(preds)[np.array([g for g in groups]) == island])
        print(f"  {island}: R²={r2_i:.3f}")

> **Resumen técnico (trazabilidad):** A partir de las observaciones individuales del dataset Palmer Penguins, se combinan descriptivos jerárquicos, comparaciones entre grupos, partición de varianza (ICC/ANOVA/modelos mixtos según especificación) y, cuando se ejecuta, validación por grupos para evaluar cuánto de la variabilidad morfométrica se asocia a especie, isla, sexo y año. La interpretación debe reconocer el diseño incompleto (confusión especie–isla) y el carácter observacional de los datos.

## 5. Lo que un ecólogo de campo sabe que el modelo no captura (¼ cuartilla)

In [ ]:
# Limitaciones explícitas
from narrative import limitacion
from IPython.display import display

display(limitacion(
    "Diseño observacional: isla y especie confundidos (no todas spp en todas islas). Biscoe=100% Gentoo, Torgersen=100% Adelie → imposible separar efecto isla puro de efecto especie.",
    "alta",
    "Modelar como crossed random effects donde sea posible; reconocer diseño incompleto y reportar ICC condicionales. En futuro: muestreo estratificado isla×especie."
))

display(limitacion(
    "Muestreo solo 3 años (2007-2009): variabilidad interanual subestimada; ICC año <1% puede ser artefacto de corto plazo.",
    "media",
    "Incorporar datos LTER 2010-2023 si disponibles; modelo con year como random slope para captura tendencia temporal."
))

display(limitacion(
    "Sexo missing not at random (11 obs, 2.9%): difícil sexar en campo sin manipulación invasiva → posible sesgo si sexo correlacionado con morfología.",
    "media",
    "Imputación múltiple con modelo predictivo (MICE) + sensitivity analysis comparando complete-case vs imputado."
))

display(limitacion(
    "Unidad de análisis = individuo adulto reproductor en colonia accesible; no representa población total (juveniles, no reproductores, colonias remotas).",
    "baja",
    "Generalizar con cautela a nivel poblacional; complementar con datos de conteo aéreo/satelital (Fretwell et al.)"
))

## 6. Cierre argumental y trazabilidad (dato → método → hallazgo → límite)

**Desarrollo argumental (sustantivo):** La unidad de análisis es el individuo medido en campo, con etiquetas de especie, isla, sexo y año. Esta estructura jerárquica y no balanceada determina el análisis: primero se describen niveles y combinaciones observadas, luego se cuantifica cuánta variabilidad se atribuye a cada nivel (ICC/efectos/modelos) y finalmente se evalúa la sensibilidad de la inferencia al esquema de muestreo (p. ej. validación dejando una isla fuera). La narrativa avanza porque cada paso reduce una posible explicación alternativa (¿es la isla o la especie?, ¿cuánto añade el sexo?, ¿qué parte es individual?).

**Trazabilidad:** mediciones morfométricas + factores → desagregación/tabla de diseño → estimaciones de varianza y efectos → contraste con limitaciones del muestreo. Las conclusiones son válidas en el marco del dataset analizado y de sus combinaciones observadas.

### Nota de coherencia narrativa y extensión
El arquetipo drill-down jerárquico y descomposición de varianza es coherente con Palmer Penguins porque la pregunta sustantiva solo puede responderse separando niveles de organización presentes en los datos; sin esa descomposición, los promedios globales serían engañosos.

In [ ]:
# Conclusión estructurada (dinámica)
from narrative import conclusion_tecnica
from IPython.display import display
import numpy as np
import pandas as pd
from stats_helpers import icc_naked, efecto_tamano_cohen_d, t_test_welch

df_mass = df.dropna(subset=['body_mass_g'])
icc_sp_n = icc_naked(df_mass, 'body_mass_g', 'species')
icc_is_n = icc_naked(df_mass, 'body_mass_g', 'island')
icc_yr_n = icc_naked(df_mass, 'body_mass_g', 'year')

# ICC ajustado por especie (si se calculó en modelo mixto previo)
icc_adj_txt = ""
try:
    if 'icc_island_adj' in globals() and np.isfinite(icc_island_adj):
        icc_adj_txt = f" Tras ajustar por especie (efecto fijo) con intercepto aleatorio por isla, la ICC asociada a isla se estima en {float(icc_island_adj):.3f} (según el modelo ejecutado)."
except Exception:
    pass

# Sexo
sex_txt = ""
df_sex = df.dropna(subset=['sex', 'body_mass_g'])
try:
    if len(df_sex) and df_sex['sex'].nunique() >= 2:
        male = df_sex[df_sex.sex == 'MALE']['body_mass_g'].values
        female = df_sex[df_sex.sex == 'FEMALE']['body_mass_g'].values
        d_sex = efecto_tamano_cohen_d(male, female)
        t_sex = t_test_welch(male, female)
        sex_txt = f" Dimorfismo sexual observado: Δ(media machos-hembras)={t_sex['diff']:.1f} g, Cohen's d={d_sex:.3f}."
except Exception:
    pass

# Medias ilustrativas por isla/especie (datos)
means_txt = ""
try:
    gmeans = df_mass.groupby(['island','species'])['body_mass_g'].mean().round(1)
    # mencionar algunos extremos presentes
    top = gmeans.sort_values(ascending=False).head(1)
    bot = gmeans.sort_values(ascending=True).head(1)
    means_txt = f" Ejemplos observados en los datos: mayor media en {top.index[0][0]}/{top.index[0][1]} (~{float(top.values[0]):.1f} g), menor en {bot.index[0][0]}/{bot.index[0][1]} (~{float(bot.values[0]):.1f} g)."
except Exception:
    pass

# LOO CV
cv_txt = ""
try:
    if 'rmse' in globals() and 'r2' in globals():
        cv_txt = f" Validación leave-one-island-out (según especificación ejecutada): RMSE={float(rmse):.2f} g, R²={float(r2):.3f}."
except Exception:
    pass

respuesta = (
    f"En el dataset analizado, la variabilidad de la masa corporal se asocia principalmente a la especie (ICC naked={icc_sp_n:.3f}) en comparación con isla ({icc_is_n:.3f}) y año ({icc_yr_n:.3f})."
    + icc_adj_txt + sex_txt + means_txt + cv_txt +
    " La interpretación considera el diseño incompleto especie×isla, que limita separar completamente efectos de contexto y taxonomía."
)

display(conclusion_tecnica(
    respuesta=respuesta,
    siguiente_paso="Ampliar el análisis con series temporales más largas y, cuando sea posible, diseños o modelos que permitan separar mejor efectos espaciales y específicos (p. ej. datos adicionales por isla/especie, modelos jerárquicos más ricos) manteniendo la trazabilidad entre nivel de análisis y conclusión.",
    reproducibilidad="Reproducible en entorno núcleo con `data/raw/penguins.csv`, `requirements.txt` y ejecución del notebook/build. Los resultados dependen de la especificación y del tratamiento de valores faltantes aplicado en el flujo."
))
